# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarBabar02/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents the daily performance of one content page for one client.

## Time Window

For this assignment, I will use data from March 2026 (2026-03). This is a mid-panel month that helps avoid using the final outcome month during analysis.

The verification queries below confirm the selected time window and the unit of analysis.

In [1]:
!pip -q install duckdb huggingface_hub

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face secret configured successfully.")

Hugging Face secret configured successfully.


In [3]:
rel = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {rel}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
"""

result = con.sql(query).df()
result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [4]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {rel}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
df = con.sql(f"""
SELECT *
FROM {rel}
LIMIT 5
""").df()
df.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

## Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

These fields are available before the ranking decision and can be used as model features.

## Label / Proxy
- Content refresh priority (ranking target)

The warehouse dataset does not contain a direct refresh label, so the ranking target will be based on observed search performance.

## Context
- report_date
- client_hash_id
- content_hash_id

These fields identify the record and are used for grouping, filtering, and joining, not for model training.

## Excluded
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

**Why?**

These fields describe data availability and system status rather than page performance, so they are used for filtering instead of prediction.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

# Query 1 — Grain

In [6]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {rel}
WHERE DATE_TRUNC('month', report_date)=DATE '2026-03-01'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*)>1
LIMIT 5
"""
con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


# **Query 2 — Counts**

In [ ]:
query = f"""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {rel}
WHERE DATE_TRUNC('month',report_date)=DATE '2026-03-01'
"""

con.sql(query).df()

# **Query 3 — Missing Values**

In [ ]:
query = f"""
SELECT
AVG(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_gsc_impressions,
AVG(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_gsc_clicks,
AVG(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_gsc_avg_position,
AVG(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS missing_ga4_pageviews,
AVG(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS missing_ga4_sessions
FROM {rel}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
"""

con.sql(query).df()

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.